## WebScraping the CTA Website for RAG pipeline

### Installing libraries and packages

In [ ]:
!pip install spacy pymupdf requests beautifulsoup4 langchain chromadb curl_cffi tiktoken
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Fou

I will split up the static and the dynamic pages on the CTA website. We will first start scraping data from the static webpages

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import spacy
import fitz  # pymupdf
from urllib.parse import urljoin, urlparse
from collections import Counter

SCRAPER_API_KEY = "2c9b18ed1042a6a0bceef8f45723d331"
BASE_URL = "https://www.transitchicago.com"
SCRAPER_ENDPOINT = "https://api.scraperapi.com"

nlp = spacy.load("en_core_web_sm")

TARGET_PAGES = {
    # --- Travel Information ---
    "travel_info_hub":          "/travel-information/",
    "getting_around":           "/travel-information/getting-around/",
    "rail_station_info":        "/travel-information/rail-station/",
    "howto_finding_your_way":   "/howto/finding-your-way/",
    "howto_paying_fare":        "/howto/pay-for-your-fare/",
    "accessibility_main":       "/accessibility/",
    "accessibility_features":   "/accessibility/features/",
    "accessibility_faq":        "/accessibility/faq/",
    "safety_security":          "/safety/",
    "rules_code_of_conduct":    "/rules/",
    "policies_practices":       "/policies-practices/",
    "bike_and_ride":            "/bikeandride/",
    # --- Fares ---
    "fares_hub":                "/fares/",
    "ventra":                   "/ventra/",
    "ventra_benefits":          "/ventrabenefits/",
    "passes":                   "/passes/",
    "reduced_fare_programs":    "/reduced-fare-programs/",
    "student_reduced_fare":     "/students/",
    "where_to_buy":             "/fares/where-to-buy/",
    # --- Footer additions ---
    "getting_help":             "/getting-help/",
    "ventra_cards_accounts":    "/ventra/cards/",
    "ventra_app":               "/ventra/app/",
    "facts_at_a_glance":        "/facts/",
    "code_of_conduct":          "/code-of-conduct/",
    "people_with_disabilities": "/disabilities/",
    "grade_high_school":        "/students/grade-high-school/",
    "college_students":         "/students/college/",
    "parents_with_children":    "/parents/",
}

NOISE_TAGS = ["script", "style", "nav", "footer", "header", "noscript", "iframe"]

# URLs/patterns to skip when following sublinks
SKIP_URL_PATTERNS = [
    "/alerts/", "/traintracker/", "/bustracker/",   # live/dynamic pages
    "/news/", "/press-releases/",                   # time-sensitive content
    "/jobs/", "/employment/",                       # not rider-facing
    "/governance/", "/board/", "/meetings/",        # admin content
    "javascript:", "mailto:", "tel:",               # non-http links
    "#",                                            # anchor-only links
    "cdn-cgi/"
]


# ── Text Cleaning

def clean_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(NOISE_TAGS):
        tag.decompose()
    raw = soup.get_text(separator="\n", strip=True)
    lines = [line.strip() for line in raw.splitlines() if line.strip()]
    deduped = [lines[0]] if lines else []
    for line in lines[1:]:
        if line != deduped[-1]:
            deduped.append(line)
    return "\n".join(deduped)


# ── Chunking

def chunk_text_by_sentences(text, source, sentences_per_chunk=8, overlap=2):
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]
    if not sentences:
        return []

    chunks = []
    step = sentences_per_chunk - overlap
    for i, start in enumerate(range(0, len(sentences), step)):
        chunk_sentences = sentences[start : start + sentences_per_chunk]
        if not chunk_sentences:
            break
        chunks.append({
            "chunk_id":     f"{source['source_id']}__chunk{i}",
            "source_id":    source["source_id"],
            "source_title": source["source_title"],
            "source_url":   source.get("source_url", ""),
            "source_type":  source.get("source_type", "scraped_web"),
            "text":         " ".join(chunk_sentences),
            "metadata": {
                "scraped_from":       source.get("source_url", ""),
                "sentence_count":     len(chunk_sentences),
                "start_sentence_idx": start,
            },
        })
    return chunks

# ── PDF Extraction

def extract_pdf_text(pdf_bytes: bytes) -> str:
    """Extract and clean text from a PDF using PyMuPDF."""
    try:
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
        pages_text = []
        for page in doc:
            text = page.get_text("text")
            if text.strip():
                pages_text.append(text.strip())
        doc.close()
        full_text = "\n".join(pages_text)

        # Clean up common PDF artifacts
        lines = [line.strip() for line in full_text.splitlines() if line.strip()]
        deduped = [lines[0]] if lines else []
        for line in lines[1:]:
            if line != deduped[-1]:
                deduped.append(line)
        return "\n".join(deduped)
    except Exception as e:
        print(f"    ⚠ PDF extraction error: {e}")
        return ""


def fetch_pdf(url: str) -> bytes | None:
    """Fetch a PDF via ScraperAPI."""
    try:
        params = {"api_key": SCRAPER_API_KEY, "url": url}
        response = requests.get(SCRAPER_ENDPOINT, params=params, timeout=60)
        response.raise_for_status()
        if b"%PDF" in response.content[:10]:
            return response.content
        # Sometimes ScraperAPI wraps the PDF — try direct fetch as fallback
        direct = requests.get(url, timeout=30)
        if b"%PDF" in direct.content[:10]:
            return direct.content
    except Exception as e:
        print(f"    ⚠ PDF fetch error for {url}: {e}")
    return None


# ── Link Discovery

def should_skip(url: str) -> bool:
    """Return True if the URL should not be followed."""
    for pattern in SKIP_URL_PATTERNS:
        if pattern in url:
            return True
    return False


def discover_links(html: str, parent_url: str) -> tuple[list[str], list[str]]:
    """
    Parse a page's HTML and return:
      - subpage_urls : internal CTA HTML pages worth scraping
      - pdf_urls     : links to PDF/brochure files
    """
    soup = BeautifulSoup(html, "html.parser")
    subpage_urls = []
    pdf_urls = []

    for a_tag in soup.find_all("a", href=True):
        href = a_tag["href"].strip()
        full_url = urljoin(parent_url, href)
        parsed = urlparse(full_url)

        # Must be on transitchicago.com
        if parsed.netloc and "transitchicago.com" not in parsed.netloc:
            continue

        # Skip unwanted patterns
        if should_skip(full_url):
            continue

        # PDFs and brochures
        if full_url.lower().endswith(".pdf"):
            pdf_urls.append(full_url)

        # Internal HTML subpages
        elif parsed.scheme in ("http", "https") and parsed.path:
            subpage_urls.append(full_url)

    # Deduplicate while preserving order
    subpage_urls = list(dict.fromkeys(subpage_urls))
    pdf_urls     = list(dict.fromkeys(pdf_urls))

    return subpage_urls, pdf_urls


# ── Core Fetch

def fetch_page(url: str, render_js: bool = False) -> str | None:
    """Fetch an HTML page via ScraperAPI. Returns raw HTML or None."""
    try:
        params = {
            "api_key": SCRAPER_API_KEY,
            "url":     url,
            "render":  "true" if render_js else "false",
        }
        response = requests.get(SCRAPER_ENDPOINT, params=params, timeout=60)
        response.raise_for_status()
        if "you have been blocked" in response.text.lower():
            return None
        return response.text
    except Exception as e:
        print(f"    ⚠ Fetch error for {url}: {e}")
        return None


# ── Main Scraper

def scrape_all(target_pages: dict) -> tuple[list, list]:
    raw_docs   = []
    all_chunks = []

    # Track everything we've already visited across all pages
    visited_urls = set()
    visited_pdfs = set()

    for name, path in target_pages.items():
        parent_url = BASE_URL + path

        if parent_url in visited_urls:
            continue

        print(f"\n[{name}] {parent_url}")

        # ── Scrape the parent page
        html = fetch_page(parent_url)
        if not html:
            print(f"  ↻ Retrying with JS rendering...")
            html = fetch_page(parent_url, render_js=True)
        if not html:
            print(f"  ✗ Failed to fetch parent page")
            continue

        visited_urls.add(parent_url)
        text = clean_text(html)
        soup = BeautifulSoup(html, "html.parser")
        title = soup.title.string.strip() if soup.title else name

        if len(text) >= 100:
            print(f"  ✓ Parent page: {len(text)} chars")
            raw_docs.append({"id": name, "url": parent_url, "title": title, "text": text})
            all_chunks.extend(chunk_text_by_sentences(text, {
                "source_id":    name,
                "source_title": title,
                "source_url":   parent_url,
                "source_type":  "scraped_web",
            }))

        # ── Discover sublinks and PDFs on this page
        subpage_urls, pdf_urls = discover_links(html, parent_url)
        print(f"  → Found {len(subpage_urls)} subpages, {len(pdf_urls)} PDFs")

        # ── Scrape subpages
        for sub_url in subpage_urls:
            if sub_url in visited_urls:
                continue

            sub_html = fetch_page(sub_url)
            if not sub_html:
                continue

            visited_urls.add(sub_url)
            sub_text = clean_text(sub_html)

            if len(sub_text) < 100:
                continue

            sub_soup  = BeautifulSoup(sub_html, "html.parser")
            sub_title = sub_soup.title.string.strip() if sub_soup.title else sub_url
            sub_id    = f"{name}__{sub_url.replace(BASE_URL, '').strip('/').replace('/', '_')}"

            print(f"    ✓ Subpage: {sub_url} ({len(sub_text)} chars)")
            raw_docs.append({"id": sub_id, "url": sub_url, "title": sub_title, "text": sub_text})
            all_chunks.extend(chunk_text_by_sentences(sub_text, {
                "source_id":    sub_id,
                "source_title": sub_title,
                "source_url":   sub_url,
                "source_type":  "scraped_web",
            }))

            time.sleep(1)

        # ── Extract PDFs
        for pdf_url in pdf_urls:
            if pdf_url in visited_pdfs:
                continue

            pdf_bytes = fetch_pdf(pdf_url)
            if not pdf_bytes:
                continue

            visited_pdfs.add(pdf_url)
            pdf_text = extract_pdf_text(pdf_bytes)

            if len(pdf_text) < 100:
                continue

            pdf_name  = pdf_url.split("/")[-1].replace(".pdf", "").replace("-", "_").replace("%20", "_")
            pdf_id    = f"{name}__pdf__{pdf_name}"
            pdf_title = pdf_name.replace("_", " ").title()

            print(f"    ✓ PDF: {pdf_url} ({len(pdf_text)} chars)")
            raw_docs.append({"id": pdf_id, "url": pdf_url, "title": pdf_title, "text": pdf_text})
            all_chunks.extend(chunk_text_by_sentences(pdf_text, {
                "source_id":    pdf_id,
                "source_title": pdf_title,
                "source_url":   pdf_url,
                "source_type":  "scraped_pdf",
            }))

            time.sleep(1)

        time.sleep(1.5)

    print(f"\n── Summary ──────────────────────────────────────────")
    print(f"  Total URLs visited : {len(visited_urls)}")
    print(f"  Total PDFs scraped : {len(visited_pdfs)}")
    return raw_docs, all_chunks


# ── Run

print("Scraping CTA pages + sublinks + PDFs via ScraperAPI...\n")

raw_docs, all_chunks = scrape_all(TARGET_PAGES)

raw_docs   = [d for d in raw_docs   if len(d["text"]) >= 100]
all_chunks = [c for c in all_chunks if len(c["text"].strip()) > 0]

with open("cta_raw_docs.json", "w", encoding="utf-8") as f:
    json.dump(raw_docs, f, indent=2, ensure_ascii=False)

with open("cta_rag_chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(raw_docs)} raw documents  → cta_raw_docs.json")
print(f"Saved {len(all_chunks)} RAG chunks    → cta_rag_chunks.json")

Scraping CTA pages + sublinks + PDFs via ScraperAPI...


[travel_info_hub] https://www.transitchicago.com/travel-information/
  ✓ Parent page: 1005 chars
  → Found 152 subpages, 0 PDFs
    ✓ Subpage: https://www.transitchicago.com/accessibility/ (6313 chars)
    ✓ Subpage: https://www.transitchicago.com/contact/ (6742 chars)
    ✓ Subpage: https://www.transitchicago.com/ (2613 chars)
    ✓ Subpage: https://www.transitchicago.com/livewellworkwell/ (7045 chars)
    ✓ Subpage: https://www.transitchicago.com/schedules/ (3768 chars)
    ✓ Subpage: https://www.transitchicago.com/fares/ (4407 chars)
    ✓ Subpage: https://www.transitchicago.com/maps/ (2378 chars)
    ✓ Subpage: https://www.transitchicago.com/tracker/ (1403 chars)
    ✓ Subpage: https://www.transitchicago.com/planatrip/ (2190 chars)
    ✓ Subpage: https://www.transitchicago.com/airports/ (7538 chars)
    ✓ Subpage: https://www.transitchicago.com/brochures/ (3503 chars)
    ✓ Subpage: https://www.transitchicago.com/destinations

Apparently the CTA has a static file on the non-static/dynamic pages like "Plan a trip", etc. The CTA contains API's that links live data like train tracker and bus tracker potentially to our system but would require periodic refresh. The GTFS static feed is probably the best solution since it is a static page we just need to embed it once and it does offer some of the info from the live/dynamic pages.

In [ ]:
import requests
import zipfile
import io
import csv
import json

GTFS_URL = "https://www.transitchicago.com/downloads/sch_data/google_transit.zip"

def chunk_text(text, source, chunk_size=512, overlap=50):
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for i, start in enumerate(range(0, len(words), step)):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            break
        chunks.append({
            "chunk_id":     f"{source['source_id']}__chunk{i}",
            "source_id":    source["source_id"],
            "source_title": source["source_title"],
            "source_url":   source.get("source_url", ""),
            "source_type":  source["source_type"],
            "text":         " ".join(chunk_words),
            "metadata":     source.get("metadata", {}),
        })
    return chunks

def read_gtfs_file(z, filename):
    """Safely read a GTFS txt file from the zip, return list of row dicts."""
    if filename not in z.namelist():
        print(f"  ⚠ {filename} not found in GTFS zip — skipping")
        return []
    with z.open(filename) as f:
        reader = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8-sig"))
        return list(reader)


def load_gtfs_docs(gtfs_url):
    print("  Downloading GTFS feed...")
    response = requests.get(gtfs_url, timeout=30)
    response.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(response.content))

    print(f"  Files in GTFS zip: {z.namelist()}\n")

    all_chunks = []

    # ── routes.txt ────────────────────────────────────────────────────────────
    rows = read_gtfs_file(z, "routes.txt")
    for row in rows:
        route_name = f"{row.get('route_short_name','')} {row.get('route_long_name','')}".strip()
        text = (
            f"CTA Route: {route_name}\n"
            f"Route ID: {row.get('route_id', 'N/A')}\n"
            f"Type: {'Bus' if row.get('route_type') == '3' else 'Rail'}\n"
            f"Description: {row.get('route_desc', 'N/A')}\n"
            f"Color: #{row.get('route_color', 'N/A')}"
        )
        source = {
            "source_id":    f"gtfs_route_{row['route_id']}",
            "source_title": f"Route {route_name}",
            "source_url":   "",
            "source_type":  "gtfs_route",
            "metadata":     {"route_id": row.get("route_id"), "route_type": row.get("route_type")},
        }
        all_chunks.extend(chunk_text(text, source))
    print(f"  ✓ routes.txt      → {len(rows)} rows")

    # ── stops.txt ─────────────────────────────────────────────────────────────
    rows = read_gtfs_file(z, "stops.txt")
    for row in rows:
        wheelchair = {"0": "No info", "1": "Accessible", "2": "Not accessible"}.get(
            row.get("wheelchair_boarding", "0"), "No info"
        )
        text = (
            f"CTA Stop: {row.get('stop_name', 'N/A')}\n"
            f"Stop ID: {row.get('stop_id', 'N/A')}\n"
            f"Description: {row.get('stop_desc', 'N/A')}\n"
            f"Location: lat {row.get('stop_lat', 'N/A')}, lon {row.get('stop_lon', 'N/A')}\n"
            f"Wheelchair accessible: {wheelchair}\n"
            f"Parent station: {row.get('parent_station', 'N/A')}"
        )
        source = {
            "source_id":    f"gtfs_stop_{row['stop_id']}",
            "source_title": f"Stop: {row.get('stop_name', 'N/A')}",
            "source_url":   "",
            "source_type":  "gtfs_stop",
            "metadata": {
                "stop_id":             row.get("stop_id"),
                "stop_lat":            row.get("stop_lat"),
                "stop_lon":            row.get("stop_lon"),
                "wheelchair_boarding": row.get("wheelchair_boarding"),
            },
        }
        all_chunks.extend(chunk_text(text, source))
    print(f"  ✓ stops.txt       → {len(rows)} rows")

    # ── trips.txt ─────────────────────────────────────────────────────────────
    rows = read_gtfs_file(z, "trips.txt")
    for row in rows:
        direction = {"0": "Outbound", "1": "Inbound"}.get(row.get("direction_id", ""), "N/A")
        text = (
            f"CTA Trip ID: {row.get('trip_id', 'N/A')}\n"
            f"Route ID: {row.get('route_id', 'N/A')}\n"
            f"Service ID: {row.get('service_id', 'N/A')}\n"
            f"Direction: {direction}\n"
            f"Headsign: {row.get('trip_headsign', 'N/A')}\n"
            f"Shape ID: {row.get('shape_id', 'N/A')}"
        )
        source = {
            "source_id":    f"gtfs_trip_{row['trip_id']}",
            "source_title": f"Trip: {row.get('trip_headsign', row.get('trip_id', 'N/A'))}",
            "source_url":   "",
            "source_type":  "gtfs_trip",
            "metadata": {
                "trip_id":    row.get("trip_id"),
                "route_id":   row.get("route_id"),
                "service_id": row.get("service_id"),
            },
        }
        all_chunks.extend(chunk_text(text, source))
    print(f"  ✓ trips.txt       → {len(rows)} rows")

    # ── calendar.txt ──────────────────────────────────────────────────────────
    rows = read_gtfs_file(z, "calendar.txt")
    for row in rows:
        days = [d for d in ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"]
                if row.get(d) == "1"]
        text = (
            f"CTA Service Schedule: {row.get('service_id', 'N/A')}\n"
            f"Operates on: {', '.join(days) if days else 'N/A'}\n"
            f"Valid from: {row.get('start_date', 'N/A')} to {row.get('end_date', 'N/A')}"
        )
        source = {
            "source_id":    f"gtfs_calendar_{row['service_id']}",
            "source_title": f"Service Schedule: {row.get('service_id', 'N/A')}",
            "source_url":   "",
            "source_type":  "gtfs_calendar",
            "metadata":     {"service_id": row.get("service_id")},
        }
        all_chunks.extend(chunk_text(text, source))
    print(f"  ✓ calendar.txt    → {len(rows)} rows")

    # ── calendar_dates.txt ────────────────────────────────────────────────────
    rows = read_gtfs_file(z, "calendar_dates.txt")
    for row in rows:
        exception = {"1": "Service added", "2": "Service removed"}.get(
            row.get("exception_type", ""), "N/A"
        )
        text = (
            f"CTA Service Exception\n"
            f"Service ID: {row.get('service_id', 'N/A')}\n"
            f"Date: {row.get('date', 'N/A')}\n"
            f"Exception: {exception}"
        )
        source = {
            "source_id":    f"gtfs_caldate_{row.get('service_id')}_{row.get('date')}",
            "source_title": f"Service Exception: {row.get('service_id')} on {row.get('date')}",
            "source_url":   "",
            "source_type":  "gtfs_calendar_dates",
            "metadata": {
                "service_id":     row.get("service_id"),
                "date":           row.get("date"),
                "exception_type": row.get("exception_type"),
            },
        }
        all_chunks.extend(chunk_text(text, source))
    print(f"  ✓ calendar_dates.txt → {len(rows)} rows")

    # ── shapes.txt ────────────────────────────────────────────────────────────
    # Group shape points by shape_id into one doc per shape
    rows = read_gtfs_file(z, "shapes.txt")
    shapes = {}
    for row in rows:
        sid = row.get("shape_id")
        shapes.setdefault(sid, []).append(row)

    for shape_id, points in shapes.items():
        # Summarize rather than dumping every lat/lon point
        text = (
            f"CTA Route Shape ID: {shape_id}\n"
            f"Total points: {len(points)}\n"
            f"Start: lat {points[0].get('shape_pt_lat')}, lon {points[0].get('shape_pt_lon')}\n"
            f"End:   lat {points[-1].get('shape_pt_lat')}, lon {points[-1].get('shape_pt_lon')}"
        )
        source = {
            "source_id":    f"gtfs_shape_{shape_id}",
            "source_title": f"Route Shape: {shape_id}",
            "source_url":   "",
            "source_type":  "gtfs_shape",
            "metadata":     {"shape_id": shape_id, "num_points": len(points)},
        }
        all_chunks.extend(chunk_text(text, source))
    print(f"  ✓ shapes.txt      → {len(shapes)} shapes from {len(rows)} points")

    # ── frequencies.txt (if present) ──────────────────────────────────────────
    rows = read_gtfs_file(z, "frequencies.txt")
    for row in rows:
        text = (
            f"CTA Trip Frequency\n"
            f"Trip ID: {row.get('trip_id', 'N/A')}\n"
            f"Start time: {row.get('start_time', 'N/A')}\n"
            f"End time: {row.get('end_time', 'N/A')}\n"
            f"Headway (seconds between trips): {row.get('headway_secs', 'N/A')}"
        )
        source = {
            "source_id":    f"gtfs_freq_{row.get('trip_id')}_{row.get('start_time','').replace(':','')}",
            "source_title": f"Frequency: Trip {row.get('trip_id')}",
            "source_url":   "",
            "source_type":  "gtfs_frequency",
            "metadata": {
                "trip_id":      row.get("trip_id"),
                "start_time":   row.get("start_time"),
                "end_time":     row.get("end_time"),
                "headway_secs": row.get("headway_secs"),
            },
        }
        all_chunks.extend(chunk_text(text, source))
    if rows:
        print(f"  ✓ frequencies.txt → {len(rows)} rows")

    # ── stop_times.txt ────────────────────────────────────────────────────────
    # ⚠ This file can be millions of rows — group by trip_id to keep chunks meaningful
    rows = read_gtfs_file(z, "stop_times.txt")
    trip_stops = {}
    for row in rows:
        trip_stops.setdefault(row.get("trip_id"), []).append(row)

    for trip_id, stops in trip_stops.items():
        first = stops[0]
        last  = stops[-1]
        text = (
            f"CTA Trip Timetable: {trip_id}\n"
            f"Total stops: {len(stops)}\n"
            f"First stop ID: {first.get('stop_id')} at {first.get('departure_time', 'N/A')}\n"
            f"Last stop ID:  {last.get('stop_id')} at {last.get('arrival_time', 'N/A')}\n"
            f"Intermediate stops: {', '.join(s.get('stop_id','') for s in stops[1:-1][:10])}"
            f"{'...' if len(stops) > 12 else ''}"
        )
        source = {
            "source_id":    f"gtfs_stoptimes_{trip_id}",
            "source_title": f"Timetable: Trip {trip_id}",
            "source_url":   "",
            "source_type":  "gtfs_stop_times",
            "metadata":     {"trip_id": trip_id, "num_stops": len(stops)},
        }
        all_chunks.extend(chunk_text(text, source))
    print(f"  ✓ stop_times.txt  → {len(trip_stops)} trips from {len(rows)} rows")

    return all_chunks


# ── Run

print("Loading GTFS data...\n")
gtfs_chunks = load_gtfs_docs(GTFS_URL)

with open("cta_gtfs_chunks.json", "w", encoding="utf-8") as f:
    json.dump(gtfs_chunks, f, indent=2, ensure_ascii=False)

print(f"\n✓ Saved {len(gtfs_chunks)} GTFS chunks → cta_gtfs_chunks.json")

# Breakdown by source type
from collections import Counter
counts = Counter(c["source_type"] for c in gtfs_chunks)
print("\nBreakdown by source_type:")
for source_type, count in counts.most_common():
    print(f"  {source_type:<25} {count} chunks")

Loading GTFS data...

  Files in GTFS zip: ['agency.txt', 'calendar.txt', 'calendar_dates.txt', 'developers_license_agreement.htm', 'frequencies.txt', 'routes.txt', 'shapes.txt', 'stop_times.txt', 'stops.txt', 'transfers.txt', 'trips.txt']

  ✓ routes.txt      → 131 rows
  ✓ stops.txt       → 11164 rows
  ✓ trips.txt       → 54663 rows
  ✓ calendar.txt    → 128 rows
  ✓ calendar_dates.txt → 85 rows
  ✓ shapes.txt      → 816 shapes from 717324 points
  ✓ stop_times.txt  → 54663 trips from 3115811 rows

✓ Saved 121650 GTFS chunks → cta_gtfs_chunks.json

Breakdown by source_type:
  gtfs_trip                 54663 chunks
  gtfs_stop_times           54663 chunks
  gtfs_stop                 11164 chunks
  gtfs_shape                816 chunks
  gtfs_route                131 chunks
  gtfs_calendar             128 chunks
  gtfs_calendar_dates       85 chunks


Combining the Static webpages with the GTFS files

In [ ]:
import json
import csv
import requests
import zipfile
import io
from google.colab import files # Import files for download

GTFS_URL = "https://www.transitchicago.com/downloads/sch_data/google_transit.zip"

# ── Source 1: Scraped Static Pages ────────────────────────────────────────────

def load_scraped_docs(filepath: str) -> list[dict]:
    """Load cta_raw_docs.json and normalize into unified chunks."""
    with open(filepath, "r", encoding="utf-8") as f:
        raw_docs = json.load(f)

    all_chunks = []
    for doc in raw_docs:
        source = {
            "source_id":    doc["id"],
            "source_title": doc["title"],
            "source_url":   doc["url"],
            "source_type":  "scraped_web",
            "metadata": {
                "scraped_from": doc["url"],
            },
        }
        all_chunks.extend(chunk_text_by_sentences(doc["text"], source))

    print(f"  Scraped pages  → {len(all_chunks)} chunks from {len(raw_docs)} docs")
    return all_chunks

# ── Source 2: GTFS Static Feed ────────────────────────────────────────────────

def load_gtfs_docs(gtfs_url: str) -> list[dict]:
    """Download GTFS zip and convert key files into unified chunks."""
    print("  Downloading GTFS feed...")
    response = requests.get(gtfs_url, timeout=30)
    response.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(response.content))

    all_chunks = []

    # --- routes.txt ---
    with z.open("routes.txt") as f:
        reader = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8"))
        for row in reader:
            route_name = f"{row['route_short_name']} {row['route_long_name']}".strip()
            text = (
                f"CTA Route: {route_name}\n"
                f"Route ID: {row['route_id']}\n"
                f"Type: {'Bus' if row['route_type'] == '3' else 'Rail'}\n"
                f"Description: {row.get('route_desc', 'N/A')}\n"
                f"Color: #{row.get('route_color', 'N/A')}"
            )
            source = {
                "source_id":    f"gtfs_route_{row['route_id']}",
                "source_title": f"Route {route_name}",
                "source_url":   "",
                "source_type":  "gtfs_route",
                "metadata": {
                    "route_id":   row["route_id"],
                    "route_type": row["route_type"],
                },
            }
            all_chunks.extend(chunk_text(text, source))

    # --- stops.txt ---
    with z.open("stops.txt") as f:
        reader = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8"))
        for row in reader:
            text = (
                f"CTA Stop: {row['stop_name']}\n"
                f"Stop ID: {row['stop_id']}\n"
                f"Location: lat {row.get('stop_lat', 'N/A')}, lon {row.get('stop_lon', 'N/A')}\n"
                f"Wheelchair accessible: {row.get('wheelchair_boarding', 'unknown')}\n"
                f"Description: {row.get('stop_desc', 'N/A')}"
            )
            source = {
                "source_id":    f"gtfs_stop_{row['stop_id']}",
                "source_title": f"Stop: {row['stop_name']}",
                "source_url":   "",
                "source_type":  "gtfs_stop",
                "metadata": {
                    "stop_id":   row["stop_id"],
                    "stop_lat":  row.get("stop_lat"),
                    "stop_lon":  row.get("stop_lon"),
                    "wheelchair_boarding": row.get("wheelchair_boarding"),
                },
            }
            all_chunks.extend(chunk_text(text, source))

    # --- calendar.txt (service days) ---
    with z.open("calendar.txt") as f:
        reader = csv.DictReader(io.TextIOWrapper(f, encoding="utf-8"))
        for row in reader:
            days = [d for d in ["monday","tuesday","wednesday","thursday","friday","saturday","sunday"] if row.get(d) == "1"]
            text = (
                f"CTA Service Schedule ID: {row['service_id']}\n"
                f"Operates on: {', '.join(days)}\n"
                f"Valid from {row.get('start_date', 'N/A')} to {row.get('end_date', 'N/A')}"
            )
            source = {
                "source_id":    f"gtfs_calendar_{row['service_id']}",
                "source_title": f"Service Schedule: {row['service_id']}",
                "source_url":   "",
                "source_type":  "gtfs_calendar",
                "metadata":     {"service_id": row["service_id"]},
            }
            all_chunks.extend(chunk_text(text, source))

    print(f"  GTFS feed      → {len(all_chunks)} chunks")
    return all_chunks

# ── Combine & Save ────────────────────────────────────────────────────────────

def combine_and_save(scraped_path: str, output_path: str):
    print("\nLoading sources...")
    scraped_chunks = load_scraped_docs(scraped_path)
    gtfs_chunks    = load_gtfs_docs(GTFS_URL)

    combined = scraped_chunks + gtfs_chunks

    # Sanity check: flag any duplicate chunk_ids
    ids = [c["chunk_id"] for c in combined]
    dupes = [id for id in ids if ids.count(id) > 1]
    if dupes:
        print(f"  ⚠ Duplicate chunk_ids found: {set(dupes)}")

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(combined, f, indent=2, ensure_ascii=False)

    print(f"\n✓ Combined dataset saved → {output_path}")
    print(f"  Total chunks : {len(combined)}")
    print(f"  Scraped web  : {len(scraped_chunks)}")
    print(f"  GTFS         : {len(gtfs_chunks)}")

    # Breakdown by source_type
    from collections import Counter
    counts = Counter(c["source_type"] for c in combined)
    print("\n  Breakdown by source_type:")
    for source_type, count in counts.most_common():
        print(f"    {source_type:<20} {count} chunks")

    files.download(output_path) # Add this line to download the file

combine_and_save(
    scraped_path="cta_raw_docs.json",
    output_path="cta_combined_dataset.json",
)


Loading sources...
  Scraped pages  → 763 chunks from 169 docs
  GTFS feed      → 11423 chunks
  ⚠ Duplicate chunk_ids found: {'travel_info_hub__updates__chunk2', 'travel_info_hub__updates__chunk1', 'travel_info_hub__updates__chunk0'}

✓ Combined dataset saved → cta_combined_dataset.json
  Total chunks : 12186
  Scraped web  : 763
  GTFS         : 11423

  Breakdown by source_type:
    gtfs_stop            11164 chunks
    scraped_web          763 chunks
    gtfs_route           131 chunks
    gtfs_calendar        128 chunks


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>